# Bu Dersi Google Colab'da Çalıştır

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BILSEM-BT/Python/blob/main/16-PythonSQLiteVeritabani.ipynb)

Bu notebook GitHub üzerinde ders dokümanı olarak yayımlanır. Kodları çalıştırmak ve üzerinde denemeler yapmak için yukarıdaki **Open in Colab** butonunu kullanabilirsiniz.

### Nasıl çalışacağız?

1. **Open in Colab** butonuna tıklayın.
2. Açılan notebook'taki kod hücrelerini `▶` düğmesiyle çalıştırın.
3. Kodları değiştirerek farklı sonuçları deneyin.
4. Çalışmalarınız kendi Colab çalışma alanınızda tutulur; bu GitHub'daki ana ders dosyasını değiştirmez.

> **Önemli:** GitHub'daki bu dosya dersin ana ve değiştirilmeyen kaynağıdır. Colab'da yaptığınız değişiklikler bu dosyaya otomatik olarak yazılmaz.

---

# 16 - Python'da SQLite ile Veritabanı İşlemleri

**Niyazi Sayın BİLSEM**  
**Bilişim Teknolojileri Dersi**  
**Ders Öğretmeni: Ersin ŞANLI**

Bu derste verileri yalnızca TXT, JSON veya CSV dosyalarında tutmak yerine **veritabanında kalıcı ve düzenli biçimde saklamaya** geçiyoruz.

Kullanacağımız veritabanı sistemi **SQLite** olacaktır.

Bu dersin sonunda öğrencinin:

- veritabanı kavramını anlaması,
- tablo, satır, sütun ve birincil anahtar kavramlarını öğrenmesi,
- SQLite veritabanı oluşturabilmesi,
- tablo oluşturabilmesi,
- kayıt ekleyebilmesi,
- kayıtları okuyabilmesi,
- kayıt güncelleyebilmesi,
- kayıt silebilmesi,
- filtreleme ve sıralama yapabilmesi,
- temel SQL fonksiyonlarını kullanabilmesi,
- parametreli sorgular yazabilmesi,
- CRUD mantığını kavraması,
- SQLite verisini Pandas ve Matplotlib ile kullanabilmesi,
- Tkinter uygulamalarında veritabanı kullanmaya hazırlanması

hedeflenmektedir.


# 1. Veritabanı Nedir?

Veritabanı, verileri düzenli, kalıcı ve sorgulanabilir biçimde saklamak için kullanılan yapıdır.

Örneğin bir öğrenci takip sisteminde:

- öğrenci numarası,
- isim,
- sınıf,
- Python puanı,
- Matematik puanı

gibi bilgiler saklanabilir.

Kayıt sayısı arttıkça metin dosyalarıyla arama, güncelleme ve silme işlemleri zorlaşır. Veritabanları bu işlemleri daha düzenli hale getirir.


# 2. Temel Kavramlar

### Veritabanı
Verilerin tamamını saklayan yapı.

### Tablo
Benzer kayıtların tutulduğu bölüm.

### Sütun / Alan
Bir özelliği temsil eder.

### Satır / Kayıt
Bir varlığa ait bilgilerin tamamıdır.

### Primary Key
Her kaydı benzersiz olarak tanımlayan alandır.


# 3. SQLite Nedir?

SQLite:

- ayrı bir sunucu gerektirmez,
- tek bir `.db` dosyasında çalışır,
- Python ile birlikte gelen `sqlite3` modülüyle kullanılabilir,
- küçük ve orta ölçekli masaüstü uygulamalar için uygundur.

Örneğin:

```text
ogrenciler.db
```

dosyası bütün öğrenci veritabanını taşıyabilir.


# 4. `sqlite3` Modülü

In [ ]:
import sqlite3

print("SQLite sürümü:", sqlite3.sqlite_version)


# 5. Veritabanına Bağlanmak

In [ ]:
import sqlite3

baglanti = sqlite3.connect("ogrenciler.db")

print("Veritabanına bağlanıldı.")

baglanti.close()


`sqlite3.connect()` belirtilen dosya yoksa yeni bir SQLite veritabanı oluşturur.


# 6. Cursor Oluşturmak

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

print("Cursor hazır.")

baglanti.close()


Cursor, SQL komutlarını veritabanına göndermek için kullanılır.


# 7. Tablo Oluşturmak

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS Ogrenciler (
    Id INTEGER PRIMARY KEY AUTOINCREMENT,
    Isim TEXT NOT NULL,
    Sinif INTEGER,
    PythonPuani INTEGER,
    MatematikPuani INTEGER
)
""")

baglanti.commit()
baglanti.close()

print("Ogrenciler tablosu hazır.")


Burada:

- `CREATE TABLE` → tablo oluşturur
- `IF NOT EXISTS` → tablo zaten varsa hata vermez
- `INTEGER` → tam sayı
- `TEXT` → metin
- `PRIMARY KEY` → benzersiz anahtar
- `AUTOINCREMENT` → ID otomatik artar
- `NOT NULL` → boş bırakılamaz


# 8. `commit()` Neden Kullanılır?

Ekleme, güncelleme ve silme gibi değişikliklerin kalıcı olması için:

```python
baglanti.commit()
```

kullanılır.


# 9. İlk Kayıt: `INSERT`

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
INSERT INTO Ogrenciler (
    Isim, Sinif, PythonPuani, MatematikPuani
)
VALUES (?, ?, ?, ?)
""", ("Ali", 8, 90, 85))

baglanti.commit()
baglanti.close()

print("Kayıt eklendi.")


`?` işaretleri parametre yer tutucularıdır.

Değerleri SQL metninin içine doğrudan birleştirmek yerine parametre kullanacağız.


# 10. Birden Fazla Kayıt: `executemany()`

In [ ]:
ogrenciler = [
    ("Ayşe", 8, 95, 92),
    ("Mehmet", 7, 72, 75),
    ("Zeynep", 8, 88, 94),
    ("Can", 7, 80, 78),
    ("Elif", 8, 97, 96)
]

baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.executemany("""
INSERT INTO Ogrenciler (
    Isim, Sinif, PythonPuani, MatematikPuani
)
VALUES (?, ?, ?, ?)
""", ogrenciler)

baglanti.commit()
baglanti.close()

print("Kayıtlar eklendi.")


# 11. Bütün Kayıtları Okumak: `SELECT`

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("SELECT * FROM Ogrenciler")

kayitlar = cursor.fetchall()

for kayit in kayitlar:
    print(kayit)

baglanti.close()


# 12. Sadece Belirli Sütunları Seçmek

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("SELECT Isim, PythonPuani FROM Ogrenciler")

for kayit in cursor.fetchall():
    print(kayit)

baglanti.close()


# 13. `fetchone()`

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("SELECT * FROM Ogrenciler ORDER BY Id")

ilk_kayit = cursor.fetchone()

print(ilk_kayit)

baglanti.close()


# 14. `WHERE` ile Filtreleme

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute(
    "SELECT * FROM Ogrenciler WHERE Sinif = ?",
    (8,)
)

for kayit in cursor.fetchall():
    print(kayit)

baglanti.close()


Tek parametreli tuple yazarken:

```python
(8,)
```

sonundaki virgüle dikkat edin.


# 15. Puan Filtreleme

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
SELECT Isim, PythonPuani
FROM Ogrenciler
WHERE PythonPuani >= ?
""", (85,))

for kayit in cursor.fetchall():
    print(kayit)

baglanti.close()


# 16. Birden Fazla Koşul

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
SELECT Isim, Sinif, PythonPuani
FROM Ogrenciler
WHERE Sinif = ?
AND PythonPuani >= ?
""", (8, 85))

for kayit in cursor.fetchall():
    print(kayit)

baglanti.close()


SQL'de sık kullanılan mantıksal ifadeler:

- `AND`
- `OR`
- `NOT`


# 17. `ORDER BY` ile Sıralama

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
SELECT Isim, PythonPuani
FROM Ogrenciler
ORDER BY PythonPuani
""")

for kayit in cursor.fetchall():
    print(kayit)

baglanti.close()


# 18. Büyükten Küçüğe Sıralama

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
SELECT Isim, PythonPuani
FROM Ogrenciler
ORDER BY PythonPuani DESC
""")

for kayit in cursor.fetchall():
    print(kayit)

baglanti.close()


- `ASC` → küçükten büyüğe
- `DESC` → büyükten küçüğe


# 19. `LIMIT`

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
SELECT Isim, PythonPuani
FROM Ogrenciler
ORDER BY PythonPuani DESC
LIMIT 3
""")

for kayit in cursor.fetchall():
    print(kayit)

baglanti.close()


Bu sorgu Python puanı en yüksek ilk 3 öğrenciyi getirir.


# 20. `UPDATE` ile Güncelleme

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
UPDATE Ogrenciler
SET PythonPuani = ?
WHERE Isim = ?
""", (93, "Ali"))

baglanti.commit()
baglanti.close()

print("Kayıt güncellendi.")


# 21. Güncellenen Kaydı Kontrol Etmek

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute(
    "SELECT * FROM Ogrenciler WHERE Isim = ?",
    ("Ali",)
)

print(cursor.fetchone())

baglanti.close()


# 22. `DELETE` ile Kayıt Silmek

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
INSERT INTO Ogrenciler (
    Isim, Sinif, PythonPuani, MatematikPuani
)
VALUES (?, ?, ?, ?)
""", ("SilinecekKisi", 7, 50, 50))

baglanti.commit()

cursor.execute(
    "DELETE FROM Ogrenciler WHERE Isim = ?",
    ("SilinecekKisi",)
)

baglanti.commit()
baglanti.close()

print("Örnek kayıt silindi.")


Silme işlemlerinde `WHERE` koşulu çok önemlidir.

```sql
DELETE FROM Ogrenciler
```

bütün kayıtları silebilir.


# 23. `COUNT()` ile Kayıt Sayısı

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("SELECT COUNT(*) FROM Ogrenciler")

print("Toplam öğrenci:", cursor.fetchone()[0])

baglanti.close()


# 24. `AVG()` ile Ortalama

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("SELECT AVG(PythonPuani) FROM Ogrenciler")

print("Python ortalaması:", cursor.fetchone()[0])

baglanti.close()


# 25. `MIN()` ve `MAX()`

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
SELECT MIN(PythonPuani), MAX(PythonPuani)
FROM Ogrenciler
""")

sonuc = cursor.fetchone()

print("En düşük:", sonuc[0])
print("En yüksek:", sonuc[1])

baglanti.close()


# 26. `SUM()`

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("SELECT SUM(PythonPuani) FROM Ogrenciler")

print("Puan toplamı:", cursor.fetchone()[0])

baglanti.close()


# 27. `GROUP BY`

Sınıflara göre Python ortalaması:


In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
SELECT Sinif, AVG(PythonPuani)
FROM Ogrenciler
GROUP BY Sinif
""")

for kayit in cursor.fetchall():
    print(kayit)

baglanti.close()


# 28. Sınıflara Göre Öğrenci Sayısı

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
SELECT Sinif, COUNT(*)
FROM Ogrenciler
GROUP BY Sinif
""")

for kayit in cursor.fetchall():
    print(kayit)

baglanti.close()


# 29. SQL İçinde Hesaplama

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
SELECT
    Isim,
    PythonPuani,
    MatematikPuani,
    (PythonPuani + MatematikPuani) / 2.0 AS Ortalama
FROM Ogrenciler
""")

for kayit in cursor.fetchall():
    print(kayit)

baglanti.close()


# 30. Ortalama Puanı 85 Üzeri Olanlar

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
SELECT
    Isim,
    (PythonPuani + MatematikPuani) / 2.0 AS Ortalama
FROM Ogrenciler
WHERE (PythonPuani + MatematikPuani) / 2.0 >= ?
ORDER BY Ortalama DESC
""", (85,))

for kayit in cursor.fetchall():
    print(kayit)

baglanti.close()


# 31. `LIKE` ile Metin Arama

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
SELECT *
FROM Ogrenciler
WHERE Isim LIKE ?
""", ("%e%",))

for kayit in cursor.fetchall():
    print(kayit)

baglanti.close()


`%` joker karakterdir.

- `A%` → A ile başlayan
- `%e` → e ile biten
- `%e%` → içinde e geçen


# 32. `BETWEEN`

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
SELECT Isim, PythonPuani
FROM Ogrenciler
WHERE PythonPuani BETWEEN ? AND ?
""", (75, 90))

for kayit in cursor.fetchall():
    print(kayit)

baglanti.close()


# 33. `IN`

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
SELECT *
FROM Ogrenciler
WHERE Sinif IN (?, ?)
""", (7, 8))

for kayit in cursor.fetchall():
    print(kayit)

baglanti.close()


# 34. `NULL` Kavramı

Python'daki `None`, SQLite'ta `NULL` olarak saklanabilir.


In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute("""
INSERT INTO Ogrenciler (
    Isim, Sinif, PythonPuani, MatematikPuani
)
VALUES (?, ?, ?, ?)
""", ("EksikNot", 8, None, 80))

baglanti.commit()

cursor.execute("""
SELECT *
FROM Ogrenciler
WHERE PythonPuani IS NULL
""")

print(cursor.fetchall())

baglanti.close()


# 35. Örnek NULL Kaydını Temizlemek

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
cursor = baglanti.cursor()

cursor.execute(
    "DELETE FROM Ogrenciler WHERE Isim = ?",
    ("EksikNot",)
)

baglanti.commit()
baglanti.close()


# 36. `sqlite3.Row` ile Sütun Adına Göre Erişim

In [ ]:
baglanti = sqlite3.connect("ogrenciler.db")
baglanti.row_factory = sqlite3.Row

cursor = baglanti.cursor()

cursor.execute("SELECT * FROM Ogrenciler LIMIT 1")

kayit = cursor.fetchone()

print("İsim:", kayit["Isim"])
print("Python:", kayit["PythonPuani"])

baglanti.close()


# 37. `with` ile SQLite Kullanmak

In [ ]:
with sqlite3.connect("ogrenciler.db") as baglanti:
    cursor = baglanti.cursor()

    cursor.execute("""
    SELECT Isim, PythonPuani
    FROM Ogrenciler
    ORDER BY PythonPuani DESC
    """)

    for kayit in cursor.fetchall():
        print(kayit)


# 38. Veritabanı Hatalarını Yönetmek

In [ ]:
try:
    baglanti = sqlite3.connect("ogrenciler.db")
    cursor = baglanti.cursor()

    cursor.execute("SELECT * FROM Ogrenciler")

    print(cursor.fetchall())

except sqlite3.Error as hata:
    print("Veritabanı hatası:", hata)

finally:
    if "baglanti" in locals():
        baglanti.close()


# 39. Kayıt Ekleme Fonksiyonu

In [ ]:
def ogrenci_ekle(isim, sinif, python_puani, matematik_puani):
    with sqlite3.connect("ogrenciler.db") as baglanti:
        cursor = baglanti.cursor()

        cursor.execute("""
        INSERT INTO Ogrenciler (
            Isim, Sinif, PythonPuani, MatematikPuani
        )
        VALUES (?, ?, ?, ?)
        """, (
            isim,
            sinif,
            python_puani,
            matematik_puani
        ))

ogrenci_ekle("Duru", 7, 91, 89)

print("Fonksiyon ile kayıt eklendi.")


# 40. Listeleme Fonksiyonu

In [ ]:
def ogrencileri_listele():
    with sqlite3.connect("ogrenciler.db") as baglanti:
        cursor = baglanti.cursor()

        cursor.execute("""
        SELECT *
        FROM Ogrenciler
        ORDER BY Id
        """)

        return cursor.fetchall()

for kayit in ogrencileri_listele():
    print(kayit)


# 41. ID'ye Göre Öğrenci Bulma

In [ ]:
def ogrenci_bul(ogrenci_id):
    with sqlite3.connect("ogrenciler.db") as baglanti:
        cursor = baglanti.cursor()

        cursor.execute(
            "SELECT * FROM Ogrenciler WHERE Id = ?",
            (ogrenci_id,)
        )

        return cursor.fetchone()

print(ogrenci_bul(1))


# 42. Güncelleme Fonksiyonu

In [ ]:
def ogrenci_guncelle(ogrenci_id, yeni_python_puani):
    with sqlite3.connect("ogrenciler.db") as baglanti:
        cursor = baglanti.cursor()

        cursor.execute("""
        UPDATE Ogrenciler
        SET PythonPuani = ?
        WHERE Id = ?
        """, (
            yeni_python_puani,
            ogrenci_id
        ))

ogrenci_guncelle(1, 94)

print("Güncelleme tamamlandı.")


# 43. Silme Fonksiyonu

In [ ]:
def ogrenci_sil(ogrenci_id):
    with sqlite3.connect("ogrenciler.db") as baglanti:
        cursor = baglanti.cursor()

        cursor.execute(
            "DELETE FROM Ogrenciler WHERE Id = ?",
            (ogrenci_id,)
        )

# Güvenli olması için burada çağırmıyoruz.
# ogrenci_sil(1)


# 44. CRUD Nedir?

Veritabanı uygulamalarının dört temel işlemi:

### Create
Yeni kayıt oluşturma → `INSERT`

### Read
Kayıtları okuma → `SELECT`

### Update
Kayıt değiştirme → `UPDATE`

### Delete
Kayıt silme → `DELETE`

Bu dört işlem birlikte **CRUD** olarak adlandırılır.


# 45. Mini CRUD Modülü

In [ ]:
DB = "ogrenciler.db"

def ekle(isim, sinif, python_puani, matematik_puani):
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()
        cursor.execute("""
        INSERT INTO Ogrenciler (
            Isim, Sinif, PythonPuani, MatematikPuani
        )
        VALUES (?, ?, ?, ?)
        """, (isim, sinif, python_puani, matematik_puani))

def listele():
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()
        cursor.execute("SELECT * FROM Ogrenciler ORDER BY Id")
        return cursor.fetchall()

def guncelle(ogrenci_id, isim, sinif, python_puani, matematik_puani):
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()
        cursor.execute("""
        UPDATE Ogrenciler
        SET Isim = ?, Sinif = ?, PythonPuani = ?, MatematikPuani = ?
        WHERE Id = ?
        """, (isim, sinif, python_puani, matematik_puani, ogrenci_id))

def sil(ogrenci_id):
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()
        cursor.execute(
            "DELETE FROM Ogrenciler WHERE Id = ?",
            (ogrenci_id,)
        )


# 46. Arama Fonksiyonu

In [ ]:
def ara(kelime):
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()

        cursor.execute("""
        SELECT *
        FROM Ogrenciler
        WHERE Isim LIKE ?
        ORDER BY Isim
        """, (f"%{kelime}%",))

        return cursor.fetchall()

for kayit in ara("e"):
    print(kayit)


# 47. Genel Ortalama Fonksiyonu

In [ ]:
def genel_ortalama():
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()

        cursor.execute("""
        SELECT AVG(
            (PythonPuani + MatematikPuani) / 2.0
        )
        FROM Ogrenciler
        """)

        return cursor.fetchone()[0]

print("Genel ortalama:", genel_ortalama())


# 48. En Başarılı İlk 3 Öğrenci

In [ ]:
def en_basarili(limit=3):
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()

        cursor.execute("""
        SELECT
            Isim,
            PythonPuani,
            MatematikPuani,
            (PythonPuani + MatematikPuani) / 2.0 AS Ortalama
        FROM Ogrenciler
        ORDER BY Ortalama DESC
        LIMIT ?
        """, (limit,))

        return cursor.fetchall()

for kayit in en_basarili(3):
    print(kayit)


# 49. SQLite → Pandas

Pandas, SQL sorgusunun sonucunu doğrudan DataFrame'e aktarabilir.


In [ ]:
import pandas as pd

with sqlite3.connect("ogrenciler.db") as baglanti:
    df = pd.read_sql_query(
        "SELECT * FROM Ogrenciler",
        baglanti
    )

df


# 50. Pandas ile Veritabanı Analizi

In [ ]:
print("Python ortalaması:", df["PythonPuani"].mean())
print("Matematik ortalaması:", df["MatematikPuani"].mean())

df["Ortalama"] = (
    df["PythonPuani"] +
    df["MatematikPuani"]
) / 2

df.sort_values(
    "Ortalama",
    ascending=False
).head()


# 51. SQLite → Pandas → Matplotlib

In [ ]:
import matplotlib.pyplot as plt

plt.bar(
    df["Isim"],
    df["Ortalama"]
)

plt.title("Öğrenci Ortalamaları")
plt.xlabel("Öğrenciler")
plt.ylabel("Ortalama")
plt.ylim(0, 100)
plt.xticks(rotation=45)

plt.show()


Bu noktada:

**SQLite → Pandas → Matplotlib**

zincirini kurduk.

Veritabanındaki veriyi analiz edip grafikle gösterebiliyoruz.


# 52. Tkinter + SQLite Mantığı

Bir önceki derste Tkinter ile masaüstü arayüzü öğrendik.

Artık veri akışımız:

```text
Kullanıcı
↓
Tkinter Formu
↓
Python Fonksiyonu
↓
SQLite Veritabanı
↓
Kayıt
```

şeklinde olabilir.


# 53. Tkinter + SQLite Kayıt Örneği

Aşağıdaki örnek masaüstü ortamında Visual Studio veya VS Code ile çalıştırılmalıdır.

```python
import tkinter as tk
from tkinter import messagebox
import sqlite3

DB = "ogrenciler.db"

def kaydet():
    isim = isim_entry.get().strip()
    sinif = sinif_entry.get().strip()
    python_puani = python_entry.get().strip()
    matematik_puani = matematik_entry.get().strip()

    if not isim or not sinif or not python_puani or not matematik_puani:
        messagebox.showwarning("Eksik Bilgi", "Tüm alanları doldurun.")
        return

    try:
        sinif = int(sinif)
        python_puani = int(python_puani)
        matematik_puani = int(matematik_puani)

        if not 0 <= python_puani <= 100:
            raise ValueError

        if not 0 <= matematik_puani <= 100:
            raise ValueError

    except ValueError:
        messagebox.showerror("Hata", "Sınıf ve puanları doğru girin.")
        return

    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()

        cursor.execute(
            '''
            INSERT INTO Ogrenciler (
                Isim, Sinif, PythonPuani, MatematikPuani
            )
            VALUES (?, ?, ?, ?)
            ''',
            (isim, sinif, python_puani, matematik_puani)
        )

    messagebox.showinfo("Başarılı", "Öğrenci kaydedildi.")
```


# 54. Treeview ile Kayıt Listeleme Mantığı

Tkinter'ın `ttk.Treeview` bileşeni tablo biçimindeki kayıtları göstermek için uygundur.

Temel yapı:

```python
from tkinter import ttk

tablo = ttk.Treeview(
    pencere,
    columns=("Id", "Isim", "Sinif", "Python", "Matematik"),
    show="headings"
)

for kayit in kayitlar:
    tablo.insert("", tk.END, values=kayit)
```

Bir sonraki proje dersinde bu yapıyı ayrıntılı olarak kullanacağız.


# 55. SQL Injection ve Parametreli Sorgular

Kullanıcı verisini SQL metnine doğrudan birleştirmek güvenli değildir.

Yanlış:

```python
sql = "SELECT * FROM Ogrenciler WHERE Isim = '" + isim + "'"
```

Doğru:

```python
cursor.execute(
    "SELECT * FROM Ogrenciler WHERE Isim = ?",
    (isim,)
)
```

Parametreli sorgular daha güvenli ve daha düzenlidir.


# 56. `UNIQUE`

Tekrar etmemesi gereken bir alan için `UNIQUE` kullanılabilir.

Örneğin:

```sql
CREATE TABLE Ogrenciler (
    Id INTEGER PRIMARY KEY AUTOINCREMENT,
    OgrenciNo INTEGER UNIQUE,
    Isim TEXT NOT NULL
)
```

Böylece aynı öğrenci numarası iki kez eklenemez.


# 57. İkinci Örnek: Stok Veritabanı

In [ ]:
with sqlite3.connect("stok.db") as baglanti:
    cursor = baglanti.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS Urunler (
        Id INTEGER PRIMARY KEY AUTOINCREMENT,
        UrunAdi TEXT NOT NULL,
        Kategori TEXT,
        Fiyat REAL,
        Stok INTEGER
    )
    """)

print("Urunler tablosu hazır.")


# 58. Ürün Kayıtları Eklemek

In [ ]:
urunler = [
    ("Laptop", "Bilgisayar", 35000, 12),
    ("Tablet", "Mobil", 18000, 20),
    ("Telefon", "Mobil", 25000, 15),
    ("Kulaklık", "Aksesuar", 2500, 40)
]

with sqlite3.connect("stok.db") as baglanti:
    cursor = baglanti.cursor()

    cursor.executemany("""
    INSERT INTO Urunler (
        UrunAdi, Kategori, Fiyat, Stok
    )
    VALUES (?, ?, ?, ?)
    """, urunler)

print("Ürünler eklendi.")


# 59. Stoku Az Olan Ürünler

In [ ]:
with sqlite3.connect("stok.db") as baglanti:
    cursor = baglanti.cursor()

    cursor.execute("""
    SELECT UrunAdi, Stok
    FROM Urunler
    WHERE Stok < ?
    ORDER BY Stok
    """, (20,))

    for kayit in cursor.fetchall():
        print(kayit)


# 60. Ürünlerin Stok Değeri

In [ ]:
with sqlite3.connect("stok.db") as baglanti:
    cursor = baglanti.cursor()

    cursor.execute("""
    SELECT
        UrunAdi,
        Fiyat,
        Stok,
        Fiyat * Stok AS StokDegeri
    FROM Urunler
    ORDER BY StokDegeri DESC
    """)

    for kayit in cursor.fetchall():
        print(kayit)


# 61. Toplam Stok Değeri

In [ ]:
with sqlite3.connect("stok.db") as baglanti:
    cursor = baglanti.cursor()

    cursor.execute("""
    SELECT SUM(Fiyat * Stok)
    FROM Urunler
    """)

    print("Toplam stok değeri:", cursor.fetchone()[0])


# 62. Kategori Bazında Analiz

In [ ]:
with sqlite3.connect("stok.db") as baglanti:
    cursor = baglanti.cursor()

    cursor.execute("""
    SELECT
        Kategori,
        COUNT(*) AS UrunSayisi,
        SUM(Stok) AS ToplamStok,
        SUM(Fiyat * Stok) AS ToplamDeger
    FROM Urunler
    GROUP BY Kategori
    """)

    for kayit in cursor.fetchall():
        print(kayit)


# 63. SQLite Veritabanı Dosyası

SQLite veritabanı normal bir dosyadır:

```text
ogrenciler.db
stok.db
```

Google Colab'da çalışma ortamı geçicidir. Oturum sıfırlandığında oluşturulan `.db` dosyaları kaybolabilir.

Masaüstü uygulamalarında `.db` dosyası proje veya uygulama klasöründe kalıcı olarak saklanabilir.


# 64. Veritabanı Yedeği Almak

In [ ]:
kaynak = sqlite3.connect("ogrenciler.db")
hedef = sqlite3.connect("ogrenciler_yedek.db")

kaynak.backup(hedef)

hedef.close()
kaynak.close()

print("Yedek oluşturuldu.")


# 65. Veritabanı Tasarımında Dikkat Edilecekler

- Her tablonun amacı belli olmalıdır.
- Her sütun anlamlı bir bilgiyi temsil etmelidir.
- Uygun veri türü seçilmelidir.
- Birincil anahtar kullanılmalıdır.
- Tekrar etmemesi gereken alanlarda `UNIQUE` düşünülmelidir.
- Kullanıcı girdileri doğrulanmalıdır.
- Parametreli sorgular kullanılmalıdır.
- Silme işlemlerinde mutlaka koşul kontrol edilmelidir.


# 66. Dosya mı Veritabanı mı?

### TXT
Basit metin ve günlük kayıtları.

### JSON
Küçük yapılandırılmış veri ve ayarlar.

### CSV
Tablo biçiminde veri aktarımı ve veri analizi.

### SQLite
Ekleme, arama, güncelleme ve silme gereken gerçek uygulamalar.

Doğru veri saklama yöntemi uygulamanın ihtiyacına göre seçilmelidir.


# 67. Ders Özeti

Bu derste:

- veritabanı,
- tablo,
- sütun,
- kayıt,
- primary key,
- SQLite,
- `sqlite3`,
- `connect()`,
- `cursor()`,
- `execute()`,
- `executemany()`,
- `commit()`,
- `CREATE TABLE`,
- `INSERT`,
- `SELECT`,
- `WHERE`,
- `ORDER BY`,
- `LIMIT`,
- `UPDATE`,
- `DELETE`,
- `COUNT`,
- `AVG`,
- `MIN`,
- `MAX`,
- `SUM`,
- `GROUP BY`,
- `LIKE`,
- `BETWEEN`,
- `IN`,
- `NULL`,
- parametreli sorgular,
- `row_factory`,
- hata yönetimi,
- CRUD,
- SQLite → Pandas,
- SQLite → Matplotlib,
- Tkinter + SQLite,
- yedekleme

konularını öğrendik.


# 68. Mini Uygulamalar

1. `okul.db` oluşturun.
2. `Ogrenciler` tablosu oluşturun.
3. En az 10 öğrenci ekleyin.
4. Bütün öğrencileri listeleyin.
5. Yalnızca isim ve Python puanlarını listeleyin.
6. 8. sınıf öğrencilerini filtreleyin.
7. Python puanı 80 ve üzeri olanları bulun.
8. Puanı 70-90 arasında olanları `BETWEEN` ile bulun.
9. İsmi A ile başlayan öğrencileri bulun.
10. Öğrencileri puana göre büyükten küçüğe sıralayın.
11. En yüksek puanlı ilk 3 öğrenciyi listeleyin.
12. Bir öğrencinin puanını güncelleyin.
13. Belirli ID'ye sahip öğrenciyi silen fonksiyon yazın.
14. Toplam öğrenci sayısını bulun.
15. Python puanı ortalamasını hesaplayın.
16. En düşük ve en yüksek puanı bulun.
17. Sınıflara göre öğrenci sayısını hesaplayın.
18. Sınıflara göre Python ortalamasını hesaplayın.
19. İki ders puanından ortalama hesaplayan SQL sorgusu yazın.
20. İsim arama fonksiyonu oluşturun.
21. CRUD işlemlerini fonksiyonlara ayırın.
22. SQLite verisini Pandas DataFrame'e aktarın.
23. Pandas ile veriyi analiz edin.
24. Matplotlib ile ortalamaları grafikle gösterin.
25. SQLite tabanlı küçük stok takip uygulaması geliştirin.


# 69. Proje Görevi

Bir **Tkinter + SQLite Öğrenci Kayıt Sistemi** tasarlayın.

Uygulamada:

- öğrenci adı,
- sınıf,
- Python puanı,
- Matematik puanı,
- kayıt ekleme,
- kayıt listeleme,
- arama,
- güncelleme,
- silme,
- form doğrulama,
- try-except,
- silme onayı,
- SQLite,
- Treeview

bulunsun.

Uygulamanın veri akışı:

```text
Tkinter Arayüzü
↓
Python Fonksiyonları
↓
SQLite
```

şeklinde olsun.


# Dersin Ana Kazanımı

Bu dersin sonunda öğrencinin şu zinciri kurabilmesi hedeflenmektedir:

**Kullanıcı / Program → SQL Sorgusu → SQLite → Kayıt → Sorgulama → Sonuç**

Şimdiye kadar TXT, JSON ve CSV ile veri saklamayı gördük.

Artık:

**Create → Read → Update → Delete**

yani **CRUD** işlemlerini gerçekleştirebilen gerçek bir veritabanı altyapısına sahibiz.

Bir sonraki derste Tkinter ile SQLite'ı birleştirerek tam bir masaüstü kayıt sistemi geliştireceğiz.
